In [ ]:
# imports

import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
from IPython.display import Markdown, display


In [ ]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')


if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    



In [ ]:
# Connect to client libraries

openai = OpenAI()

ollama_url = "http://localhost:11434/v1"
ollama = OpenAI(api_key="ollama", base_url=ollama_url)

In [ ]:
models = ["gpt-5", "gpt-oss:20b", "qwen2.5-coder", "deepseek-coder-v2", "llama3.2:latest" ]

clients = {"gpt-5": openai,  "qwen2.5-coder": ollama, "deepseek-coder-v2": ollama, "gpt-oss:20b": ollama, "llama3.2:latest": ollama}

# Want to keep costs ultra-low? Replace this with models of your choice, using the examples from yesterday

In [ ]:
system_prompt = """
Your task is to identify the language that the user has passed to you and comment the code.
Respond only with the appropriate about what the code does. No need to oversimplify things and you should not make big blocks of comments.
The code should be identical, only added things should be the comments.
"""

def user_prompt_for(code):
    return f"""
Comment this code in the simplest manner possible. Follow the K.I.S.S (Keep it simple, stupid) rule while commenting the code.
Respond only with commented code in the same language.
Start the line with a comment that mentions the extension of the file which is being commented.
code to comment:
```code
{code}
```
"""

In [ ]:
def messages_for(code):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(code)}
    ]

In [ ]:
def write_output(code, ext):
    with open(code + "." + ext, "w") as f:
        f.write(code)

In [ ]:
def comment(model, code):
    client = clients[model]
    openai_reasoning_models = {"gpt-5"}
    reasoning_effort = "high" if model in openai_reasoning_models else None
    response = client.chat.completions.create(model=model, messages=messages_for(code), reasoning_effort=reasoning_effort)
    reply = response.choices[0].message.content
    return reply

In [ ]:
pi = """
import time

def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations+1):
        j = i * param1 - param2
        result -= (1/j)
        j = i * param1 + param2
        result += (1/j)
    return result

start_time = time.time()
result = calculate(200_000_000, 4, 1) * 4
end_time = time.time()

print(f"Result: {result:.12f}")
print(f"Execution Time: {(end_time - start_time):.6f} seconds")
"""

In [ ]:
languages = ["python", "c", "cpp", "javascript", "typescript", "html", "css", "sql", "shell", "r", "json", "yaml", "dockerfile", "markdown"]

def set_language(language):
    # Apply the chosen language to both panes
    return gr.Code(language=language, label=f"{language} (original)"), gr.Code(language=language, label=f"{language} (commented)")

with gr.Blocks(theme=gr.themes.Monochrome(), title="Comment your code out") as ui:
    with gr.Row():
        language = gr.Dropdown(languages, value=languages[0], label="Language")
        model = gr.Dropdown(models, value=models[0], label="Model")
        convert = gr.Button("Comment code")

    with gr.Row(equal_height=True):
        original = gr.Code(label=f"{languages[0]} (original)", value=pi, language=languages[0], lines=26)
        commented = gr.Code(label=f"{languages[0]} (commented)", value="", language=languages[0], lines=26)

    language.change(fn=set_language, inputs=[language], outputs=[original, commented])
    convert.click(fn=comment, inputs=[model, original], outputs=[commented])

ui.launch(inbrowser=True)

In [ ]:
gr.close_all()